In [4]:
import sys
import torch
import math
from xray_gaussian_rasterization_voxelization import (
    GaussianRasterizationSettings,
    GaussianRasterizer,
    GaussianVoxelizationSettings,
    GaussianVoxelizer,
)

sys.path.append("./")
from r2_gaussian.gaussian.gaussian_model import GaussianModel
from r2_gaussian.dataset.cameras import Camera
from r2_gaussian.arguments import PipelineParams

In [5]:
from r2_gaussian.dataset import Scene
from r2_gaussian.gaussian import GaussianModel, render, query, initialize_gaussian

In [6]:
from argparse import ArgumentParser
from r2_gaussian.arguments import (
    ModelParams,
    PipelineParams,
    get_combined_args,
)

In [7]:
from argparse import Namespace

In [24]:
parser = ArgumentParser(description="Testing script parameters")
model = ModelParams(parser, sentinel=True)

pipeline = PipelineParams(parser)


# Namespace で引数を手動指定
args = Namespace(
    source_path="/home/maemaeko/imari_lab/r2_gaussian/data/synthetic_dataset/cone_ntrain_75_angle_360/0_foot_cone",
    #model_path="/home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_3_angle_360-wo-densification/2_teapot_cone",
    model_path="/home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_75_angle_360/0_foot_cone",
    data_device="cuda",
    scale_min=0.0005,
    scale_max=0.5,
    eval=True
)

dataset = model.extract(args)
scene = Scene(
    dataset,
    shuffle=False
)

Reading camera 75/75 for train
Reading camera 100/100 for test
Loading Training Cameras
Loading Test Cameras


In [25]:
# Set up Gaussians
gaussians = GaussianModel(None)  # scale_bound will be loaded later
loaded_iter = initialize_gaussian(gaussians, dataset, -1)
scene.gaussians = gaussians

Loading trained model at iteration 10000
Loading from /home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_75_angle_360/0_foot_cone/point_cloud/iteration_10000/point_cloud.pickle


In [26]:
import numpy as np

def _normalize(v, eps=1e-8):
    n = np.linalg.norm(v)
    return v / (n + eps)

def look_at_c2w(camera_center, target=np.array([0., 0., 0.], dtype=np.float32),
                up=np.array([0., 1., 0.], dtype=np.float32)):
    """
    camera_center: (3,) world coords (C)
    target: (3,) world coords (look-at point), default origin
    up: (3,) world up direction
    return: c2w (4,4) camera-to-world
    """
    C = camera_center.astype(np.float32)
    T = target.astype(np.float32)

    # forward (camera +z looking direction in world)
    forward = _normalize(T - C)

    # right
    right = _normalize(np.cross(forward, up))
    # up' (re-orthogonalize)
    up2 = np.cross(right, forward)

    # c2w rotation: columns = [right, up, forward]
    c2w = np.eye(4, dtype=np.float32)
    c2w[:3, 0] = right
    c2w[:3, 1] = up2
    c2w[:3, 2] = forward
    c2w[:3, 3] = C
    return c2w

def generate_random_RT_bbox_lookat_origin(
    n_poses=120,
    bbox_min=-5.0,
    bbox_max= 5.0,
    up=np.array([0., 1., 0.], dtype=np.float32),
    min_radius=1e-3,
    seed=None,
):
    """
    Returns:
      R: (n_poses, 3, 3)  # 3DGS系でよくある保存形式: R = (w2c[:3,:3]).T
      T: (n_poses, 3)     # T = w2c[:3,3]
    """
    rng = np.random.default_rng(seed)

    Rs, Ts = [], []
    target = np.array([0., 0., 0.], dtype=np.float32)

    for _ in range(n_poses):
        # 位置を bbox 内からサンプル（原点近傍だと look-at が不安定なので避ける）
        for _try in range(1000):
            C = rng.uniform(bbox_min, bbox_max, size=(3,)).astype(np.float32)
            if np.linalg.norm(C - target) > min_radius:
                break

        c2w = look_at_c2w(C, target=target, up=up)
        w2c = np.linalg.inv(c2w)

        # あなたの既存コードに合わせた取り出し方
        R = w2c[:3, :3].T.astype(np.float32)
        T = w2c[:3, 3].astype(np.float32)

        Rs.append(R)
        Ts.append(T)

    R = np.stack(Rs, axis=0)
    T = np.stack(Ts, axis=0)
    return R, T


In [27]:
R, T = generate_random_RT_bbox_lookat_origin(up=np.array([0,0,1], np.float32))

training_cam = scene.getTrainCameras()

camera_lists = []

for i in range(len(R)):
    viewpoint_cam = training_cam[0]
    cam = Camera(
        colmap_id=viewpoint_cam.colmap_id,
        scanner_cfg=None,
        R=R[i],
        T=T[i],
        angle=viewpoint_cam.angle,
        mode=viewpoint_cam.mode,
        FoVx=viewpoint_cam.FoVx,
        FoVy=viewpoint_cam.FoVy,
        image=torch.zeros((1, 512, 512)),
        image_name="none",
        uid=1,
    )
    camera_lists.append(cam)

In [30]:
from r2_gaussian.arguments import ModelParams
from r2_gaussian.dataset import Scene
from r2_gaussian.utils.plot_utils import create_textured_camera, create_vol_mesh
from r2_gaussian.utils.graphics_utils import fov2focal
from r2_gaussian.utils.general_utils import t2a
import os.path as osp
import open3d as o3d
import matplotlib

scanner_cfg = scene.scanner_cfg
mc_thresh = 0.2
cam_scale = 1.0

vol_mesh = create_vol_mesh(
    np.load(osp.join(dataset.source_path, "vol_gt.npy")),
    np.array(scanner_cfg["offOrigin"]),
    np.array(scanner_cfg["dVoxel"]),
    np.eye(3),
    level=mc_thresh,
)

vol_coord = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=scanner_cfg["sVoxel"][0] / 2,
    origin=scanner_cfg["offOrigin"],
)

vol_bbox = o3d.geometry.OrientedBoundingBox(
    center=scanner_cfg["offOrigin"],
    R=np.eye(3),
    extent=scanner_cfg["sVoxel"],
)


vol_bbox.color = np.array([1, 0, 0])

unit_bbox = o3d.geometry.OrientedBoundingBox(
    center=[0, 0, 0], R=np.eye(3), extent=[2, 2, 2]
)
unit_bbox.color = np.array([0, 0, 1])

cams = []
cmap = matplotlib.colormaps["viridis"]
cam_scale = cam_scale
n_proj = len(camera_lists)
for i_proj, camera in enumerate(camera_lists):
    proj_name = camera.image_name
    proj_id = i_proj
    proj = t2a(camera.original_image)[0]
    K = np.array(
        [
            [fov2focal(camera.FoVx, proj.shape[1]), 0, proj.shape[1] / 2],
            [0, fov2focal(camera.FoVy, proj.shape[0]), proj.shape[0] / 2],
            [0, 0, 1],
        ]
    )
    w2c = np.eye(4)
    w2c[:3, :3] = t2a(camera.R.T)
    w2c[:3, 3] = t2a(camera.T)
    c2w = np.linalg.inv(w2c)
    DSO = np.linalg.norm(c2w[:3, 3] - np.array(scanner_cfg["offOrigin"]))
    cam = create_textured_camera(
        K,
        w2c,
        cam_scale,
        cmap(i_proj / n_proj)[:3],
        proj.shape[1],
        proj.shape[0],
        f"{proj_id:03d}",
        proj,
    )
    cams += cam

vis_assets = cams + [vol_mesh, vol_bbox, vol_coord, unit_bbox]
o3d.visualization.draw_geometries(vis_assets, mesh_show_back_face=True)


In [35]:
import numpy as np

def _normalize(v, eps=1e-8):
    n = np.linalg.norm(v)
    return v / (n + eps)

def look_at_c2w(camera_center,
                target=np.array([0., 0., 0.], dtype=np.float32),
                up=np.array([0., 1., 0.], dtype=np.float32)):
    """
    camera_center: (3,) world coords (C)
    target:       (3,) look-at point (デフォルト原点)
    up:           (3,) world up direction
    return:       c2w (4,4)
    """
    C = camera_center.astype(np.float32)
    T = target.astype(np.float32)

    # forward: カメラの +z (画面奥) の向き in world
    forward = _normalize(T - C)

    # right: x
    right = _normalize(np.cross(forward, up))
    # re-orthogonalized up
    up2 = np.cross(right, forward)

    c2w = np.eye(4, dtype=np.float32)
    c2w[:3, 0] = right
    c2w[:3, 1] = up2
    c2w[:3, 2] = forward
    c2w[:3, 3] = C
    return c2w


def rotate_vec_axis_angle(v, axis, angle):
    """
    v    : (3,) 回転させたいベクトル
    axis : (3,) 回転軸（単位ベクトル）
    angle: [rad]
    """
    v = v.astype(np.float32)
    axis = _normalize(axis.astype(np.float32))
    cos_t = np.cos(angle)
    sin_t = np.sin(angle)

    # Rodrigues の回転公式
    return (v * cos_t
            + np.cross(axis, v) * sin_t
            + axis * np.dot(axis, v) * (1.0 - cos_t))

def generate_random_RT_pairs_pm1deg(
    n_poses=120,
    bbox_min=-5.0,
    bbox_max= 5.0,
    up=np.array([0., 1., 0.], dtype=np.float32),
    min_radius=1e-3,
    seed=None,
):
    """
    ランダムなカメラ位置 C を bbox 内からサンプルして、
    - base カメラ: C を注視点(0,0,0)に向けた姿勢
    - offset カメラ: C を原点中心の回転で ±1° だけずらした位置 C'
                     を注視点(0,0,0)に向けた姿勢
    を作る。

    戻り値:
      R_base:   (N, 3, 3)
      T_base:   (N, 3)
      R_offset: (N, 3, 3)
      T_offset: (N, 3)
    ここで R, T はあなたのコードに合わせて:
      w2c = inv(c2w)
      R = w2c[:3,:3].T
      T = w2c[:3,3]
    となっています。
    """
    rng = np.random.default_rng(seed)
    target = np.array([0., 0., 0.], dtype=np.float32)

    R_base_list   = []
    T_base_list   = []
    R_offset_list = []
    T_offset_list = []

    for _ in range(n_poses):
        # 1) bbox 内から C_base をサンプル（原点に近すぎる場合は再サンプル）
        for _try in range(1000):
            C_base = rng.uniform(bbox_min, bbox_max, size=(3,)).astype(np.float32)
            if np.linalg.norm(C_base - target) > min_radius:
                break

        # 2) C_base に直交するランダム軸を作る
        for _try in range(10):
            rand = rng.normal(size=(3,))
            axis = np.cross(C_base, rand)
            if np.linalg.norm(axis) > 1e-6:
                break
        else:
            # まれに全部ダメだったときの保険
            axis = np.array([0., 1., 0.], dtype=np.float32)

        # 3) ±1° を rad にして決める
        sign = rng.choice(np.array([-1.0, 1.0]))
        angle = sign * np.deg2rad(1.0)

        # 4) C_base を原点中心に回転 → C_offset
        C_offset = rotate_vec_axis_angle(C_base, axis, angle)

        # 5) それぞれ look-at 原点で c2w を作る
        c2w_base   = look_at_c2w(C_base,   target=target, up=up)
        c2w_offset = look_at_c2w(C_offset, target=target, up=up)

        # 6) w2c にしてから、あなたの形式の R, T に変換
        w2c_base   = np.linalg.inv(c2w_base)
        w2c_offset = np.linalg.inv(c2w_offset)

        R_base   = w2c_base[:3, :3].T.astype(np.float32)
        T_base   = w2c_base[:3, 3].astype(np.float32)
        R_offset = w2c_offset[:3, :3].T.astype(np.float32)
        T_offset = w2c_offset[:3, 3].astype(np.float32)

        R_base_list.append(R_base)
        T_base_list.append(T_base)
        R_offset_list.append(R_offset)
        T_offset_list.append(T_offset)

    R_base   = np.stack(R_base_list,   axis=0)
    T_base   = np.stack(T_base_list,   axis=0)
    R_offset = np.stack(R_offset_list, axis=0)
    T_offset = np.stack(T_offset_list, axis=0)
    return R_base, T_base, R_offset, T_offset

In [37]:
# ランダムなペアを作る
R0, T0, R1, T1 = generate_random_RT_pairs_pm1deg(
    n_poses=64,
    bbox_min=-5.0,
    bbox_max= 5.0,
    up=np.array([0., 0., 1.], dtype=np.float32),
    seed=0,
)

training_cam = scene.getTrainCameras()

camera_lists = []

for i in range(len(R0)):
    viewpoint_cam = training_cam[0]
    cam_0 = Camera(
        colmap_id=viewpoint_cam.colmap_id,
        scanner_cfg=None,
        R=R0[i],
        T=T0[i],
        angle=viewpoint_cam.angle,
        mode=viewpoint_cam.mode,
        FoVx=viewpoint_cam.FoVx,
        FoVy=viewpoint_cam.FoVy,
        image=torch.zeros((1, 512, 512)),
        image_name="none",
        uid=1,
    )
    camera_lists.append(cam_0)
    cam_1 = Camera(
        colmap_id=viewpoint_cam.colmap_id,
        scanner_cfg=None,
        R=R1[i],
        T=T1[i],
        angle=viewpoint_cam.angle,
        mode=viewpoint_cam.mode,
        FoVx=viewpoint_cam.FoVx,
        FoVy=viewpoint_cam.FoVy,
        image=torch.zeros((1, 512, 512)),
        image_name="none",
        uid=1,
    )
    camera_lists.append(cam_1)


In [38]:
from r2_gaussian.arguments import ModelParams
from r2_gaussian.dataset import Scene
from r2_gaussian.utils.plot_utils import create_textured_camera, create_vol_mesh
from r2_gaussian.utils.graphics_utils import fov2focal
from r2_gaussian.utils.general_utils import t2a
import os.path as osp
import open3d as o3d
import matplotlib

scanner_cfg = scene.scanner_cfg
mc_thresh = 0.2
cam_scale = 1.0

vol_mesh = create_vol_mesh(
    np.load(osp.join(dataset.source_path, "vol_gt.npy")),
    np.array(scanner_cfg["offOrigin"]),
    np.array(scanner_cfg["dVoxel"]),
    np.eye(3),
    level=mc_thresh,
)

vol_coord = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=scanner_cfg["sVoxel"][0] / 2,
    origin=scanner_cfg["offOrigin"],
)

vol_bbox = o3d.geometry.OrientedBoundingBox(
    center=scanner_cfg["offOrigin"],
    R=np.eye(3),
    extent=scanner_cfg["sVoxel"],
)


vol_bbox.color = np.array([1, 0, 0])

unit_bbox = o3d.geometry.OrientedBoundingBox(
    center=[0, 0, 0], R=np.eye(3), extent=[2, 2, 2]
)
unit_bbox.color = np.array([0, 0, 1])

cams = []
cmap = matplotlib.colormaps["viridis"]
cam_scale = cam_scale
n_proj = len(camera_lists)
for i_proj, camera in enumerate(camera_lists):
    proj_name = camera.image_name
    proj_id = i_proj
    proj = t2a(camera.original_image)[0]
    K = np.array(
        [
            [fov2focal(camera.FoVx, proj.shape[1]), 0, proj.shape[1] / 2],
            [0, fov2focal(camera.FoVy, proj.shape[0]), proj.shape[0] / 2],
            [0, 0, 1],
        ]
    )
    w2c = np.eye(4)
    w2c[:3, :3] = t2a(camera.R.T)
    w2c[:3, 3] = t2a(camera.T)
    c2w = np.linalg.inv(w2c)
    DSO = np.linalg.norm(c2w[:3, 3] - np.array(scanner_cfg["offOrigin"]))
    cam = create_textured_camera(
        K,
        w2c,
        cam_scale,
        cmap(i_proj / n_proj)[:3],
        proj.shape[1],
        proj.shape[0],
        f"{proj_id:03d}",
        proj,
    )
    cams += cam

vis_assets = cams + [vol_mesh, vol_bbox, vol_coord, unit_bbox]
o3d.visualization.draw_geometries(vis_assets, mesh_show_back_face=True)


/home/maemaeko/imari_lab/r2_gaussian/r2_gaussian/utils/plot_utils.py:483: RuntimeWarning: invalid value encountered in divide
  image = (image / np.max(image) * 255).astype(np.uint8)
/home/maemaeko/imari_lab/r2_gaussian/r2_gaussian/utils/plot_utils.py:483: RuntimeWarning: invalid value encountered in cast
  image = (image / np.max(image) * 255).astype(np.uint8)
